In [ ]:
from src.agents import ABPruningAgent, MiniMaxAgent
import gymnasium as gym
import time
import cProfile
from src.agents.ab_pruning.ab_agent import mine

In [ ]:
BOARD_SIZE = 7

env = gym.make('gymnasium_env/Blokus-v0', board_size=BOARD_SIZE, num_players=2, render_scale=10, testing_mode=False)
env = env.unwrapped

In [ ]:
from src.agents.ab_pruning.heuristics import *
import pickle
orders = []

WA = 1
WB = 2
DEPTH = 5, 5

def order_fn(env, obs, action, depth, i_, j_, depth_level_):
    # print(i, j, depth_level)
    if depth < depth_level_:
        return maximise_our_expanders_difference(env, obs, action, i_, j_)
    else:
        return level(env, obs, action)
    
for i in range(WA):
    for j in range(WB):
        if i == 0 and j == 0:
            continue
        for depth_level in range(DEPTH[0], DEPTH[1] + 1):
            name = f"level_{depth_level}_WA_{i}_WB_{j}"
            orders.append((name, lambda x, i=i, j=j, depth_level=depth_level: order_fn(*x, i, j, depth_level)))
# name = name = f"level_{0}_WA_{0}_WB_{1}"
orders.append((name, lambda x, i=0, j=1, depth_level=0: order_fn(*x, i, j, depth_level)))
logs = {}

# orders.append(("level", lambda x: maximise_our_expanders_difference(*x[:-1], 0, 1)))
for name, order in orders:
    print(name)
    abpruning_agent = ABPruningAgent(board_size=BOARD_SIZE, depth=-1, use_cache=False, sorted_order=order)
    # get_action2 = MiniMaxAgent(board_size=BOARD_SIZE).get_action
    obs, info = env.reset()
    # env.render_mode = "human"
    # obs, _, _, _, _ = env.step(28)
    # action1 = env._tuple_to_action((4, 4, 'F', 7))
    # obs, _, _, _, _ = env.step(action1)
    # action2 = env._tuple_to_action((0, 3, 'Y', 4))
    # obs, _, _, _, _ = env.step(action2)
    # action3 = env._tuple_to_action((4, 1, 'T5', 1))
    # obs, _, _, _, _ = env.step(action3)
    action, timed = abpruning_agent.get_action(env, obs)
    logs[name] = abpruning_agent.log, timed
    # print(action)
    # print(abpruning_agent.num_pruned)

with open('logs3.pkl', 'wb') as f:
    pickle.dump(logs, f)
# cProfile.run('main()')

# print(action)
# action = get_action2(env, obs)

In [ ]:
print(action)
# action2 = env._tuple_to_action((2, 3, 'N', 0))
    # mine(env, obs, action2, print_=True)
    # action2 = env._tuple_to_action((0, 3, 'Y', 4))
    # env.step(action2)
    # action3 = env._tuple_to_action((4, 1, 'T5', 1))
    # env.step(action3)
    # action4 = env._tuple_to_action((2, 3, 'L5', 1))
    # env.step(action4)
    # action5 = env._tuple_to_action((2, 0, 'Y', 1))
    # env.step(action5)
    # action6 = 1752
    # env.step(action6)
    # action7 = 1456
    # env.step(action7)
    # action8 = 4187
    # env.step(action8)
    # action9 = 911
    # obs1, rewrd, term, trunc, info1 = env.step(action9)
    # env.render()
# print(env._action_to_tuple(action))

# # abpruning_agent.save_cache()

In [ ]:
import numpy as np
import pickle
import matplotlib.pyplot as plt

# Assuming self.log is a list of tuples (num_pruned, visited_states, pruned_percentage)

# Plot the data
plt.figure(figsize=(20, 12))
K = 10

with open('logs3.pkl', 'rb') as f:
    logs = pickle.load(f)

max_length = 0
max_name = None
INF = 1e9
# min_length = [(INF) for _ in range(5)]
# min_name = ["" for _ in range(5)]
for name, log in logs.items():
    log, timed = log[0], log[1]
    # print(log)
    # print(name, timed)
    if max_length < len(log):
        max_length = len(log)
        max_name = name
    # if min_length[int(name[6])] > timed:
    #     min_length[int(name[6])] = timed
    #     min_name[int(name[6])] = name
    num_pruned = np.array([entry[0] for entry in log])
    visited_states = np.array([entry[1] for entry in log])
    pruned_percentage = np.array([entry[2] for entry in log])
    # plt.scatter(num_pruned[::K], pruned_percentage[::K], marker='.')
    plt.plot(num_pruned[::K], pruned_percentage[::K], label=f"{name}, t:{timed:.2f}s", linewidth=0.5)
print(max_name)
# for i in range(1, 5):
#     print(min_length[i], min_name[i])
plt.xlabel('Visited States')
plt.legend()
plt.ylabel('Pruned Percentage')
plt.title('Pruned Percentage vs Visited States')
plt.grid(True)
plt.savefig('pruned_percentage_vs_visited_states.png')
plt.show()